In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit, upper, concat, year, current_date

spark = SparkSession.builder.appName("PysparkTrans").master("local[*]").getOrCreate()
print("Spark is running:", spark.version)

data = [
    (1, "alice", "engineering", 95000, "2022-03-15", "active"),
    (2, "bob", "marketing", 72000, "2021-07-01", "active"),
    (3, "carol", "engineering", 110000, "2019-11-20", "active"),
    (4, "dave", "sales", 68000, "2023-01-10", "inactive"),
    (5, "eve", "engineering", 125000, "2018-06-05", "active"),
    (6, "frank", "marketing", 78000, "2022-09-12", "inactive"),
]

df = spark.createDataFrame(
    data,
    schema=["emp_id", "name", "department", "salary", "hire_date", "status"]
)

Spark is running: 4.2.0


In [3]:
spark

In [4]:
df.select("name", "salary").show()

+-----+------+
| name|salary|
+-----+------+
|alice| 95000|
|  bob| 72000|
|carol|110000|
| dave| 68000|
|  eve|125000|
|frank| 78000|
+-----+------+



In [6]:
df.select(df.name, df.salary).show()

+-----+------+
| name|salary|
+-----+------+
|alice| 95000|
|  bob| 72000|
|carol|110000|
| dave| 68000|
|  eve|125000|
|frank| 78000|
+-----+------+



In [8]:
df.select(col("name"), col("salary")).show()

+-----+------+
| name|salary|
+-----+------+
|alice| 95000|
|  bob| 72000|
|carol|110000|
| dave| 68000|
|  eve|125000|
|frank| 78000|
+-----+------+



In [10]:
df.select(
    col("emp_id"),
    upper(col("name")).alias("name_upper"),
    (col("salary") / 12).alias("monthly_salary"),
    concat(col("department"), lit("_team")).alias("team_label"),
).show()

+------+----------+------------------+----------------+
|emp_id|name_upper|    monthly_salary|      team_label|
+------+----------+------------------+----------------+
|     1|     ALICE| 7916.666666666667|engineering_team|
|     2|       BOB|            6000.0|  marketing_team|
|     3|     CAROL| 9166.666666666666|engineering_team|
|     4|      DAVE| 5666.666666666667|      sales_team|
|     5|       EVE|10416.666666666666|engineering_team|
|     6|     FRANK|            6500.0|  marketing_team|
+------+----------+------------------+----------------+



In [11]:
# Single condition filtering
df.filter(col("department") == "engineering").show()

+------+-----+-----------+------+----------+------+
|emp_id| name| department|salary| hire_date|status|
+------+-----+-----------+------+----------+------+
|     1|alice|engineering| 95000|2022-03-15|active|
|     3|carol|engineering|110000|2019-11-20|active|
|     5|  eve|engineering|125000|2018-06-05|active|
+------+-----+-----------+------+----------+------+



In [12]:
# AND — use &
df.filter(
    (col("department") == "engineering") & (col("salary") > 100000)
).show()

+------+-----+-----------+------+----------+------+
|emp_id| name| department|salary| hire_date|status|
+------+-----+-----------+------+----------+------+
|     3|carol|engineering|110000|2019-11-20|active|
|     5|  eve|engineering|125000|2018-06-05|active|
+------+-----+-----------+------+----------+------+



In [13]:
# OR — use |
df.filter(
    (col("department") == "engineering") | (col("department") == "marketing")
).show()

+------+-----+-----------+------+----------+--------+
|emp_id| name| department|salary| hire_date|  status|
+------+-----+-----------+------+----------+--------+
|     1|alice|engineering| 95000|2022-03-15|  active|
|     2|  bob|  marketing| 72000|2021-07-01|  active|
|     3|carol|engineering|110000|2019-11-20|  active|
|     5|  eve|engineering|125000|2018-06-05|  active|
|     6|frank|  marketing| 78000|2022-09-12|inactive|
+------+-----+-----------+------+----------+--------+



In [14]:
# NOT — use ~
df.filter(~(col("status") == "inactive")).show()

+------+-----+-----------+------+----------+------+
|emp_id| name| department|salary| hire_date|status|
+------+-----+-----------+------+----------+------+
|     1|alice|engineering| 95000|2022-03-15|active|
|     2|  bob|  marketing| 72000|2021-07-01|active|
|     3|carol|engineering|110000|2019-11-20|active|
|     5|  eve|engineering|125000|2018-06-05|active|
+------+-----+-----------+------+----------+------+



In [15]:
# IN list
df.filter(col("department").isin("engineering", "marketing")).show()

+------+-----+-----------+------+----------+--------+
|emp_id| name| department|salary| hire_date|  status|
+------+-----+-----------+------+----------+--------+
|     1|alice|engineering| 95000|2022-03-15|  active|
|     2|  bob|  marketing| 72000|2021-07-01|  active|
|     3|carol|engineering|110000|2019-11-20|  active|
|     5|  eve|engineering|125000|2018-06-05|  active|
|     6|frank|  marketing| 78000|2022-09-12|inactive|
+------+-----+-----------+------+----------+--------+



In [16]:
# NULL checks
df.filter(col("name").isNotNull()).show()
df.filter(col("name").isNull()).show()

+------+-----+-----------+------+----------+--------+
|emp_id| name| department|salary| hire_date|  status|
+------+-----+-----------+------+----------+--------+
|     1|alice|engineering| 95000|2022-03-15|  active|
|     2|  bob|  marketing| 72000|2021-07-01|  active|
|     3|carol|engineering|110000|2019-11-20|  active|
|     4| dave|      sales| 68000|2023-01-10|inactive|
|     5|  eve|engineering|125000|2018-06-05|  active|
|     6|frank|  marketing| 78000|2022-09-12|inactive|
+------+-----+-----------+------+----------+--------+

+------+----+----------+------+---------+------+
|emp_id|name|department|salary|hire_date|status|
+------+----+----------+------+---------+------+
+------+----+----------+------+---------+------+



In [18]:
# String patterns
df.filter(col("name").startswith("a")).show()
df.filter(col("name").contains("ar")).show()

+------+-----+-----------+------+----------+------+
|emp_id| name| department|salary| hire_date|status|
+------+-----+-----------+------+----------+------+
|     1|alice|engineering| 95000|2022-03-15|active|
+------+-----+-----------+------+----------+------+

+------+-----+-----------+------+----------+------+
|emp_id| name| department|salary| hire_date|status|
+------+-----+-----------+------+----------+------+
|     3|carol|engineering|110000|2019-11-20|active|
+------+-----+-----------+------+----------+------+



In [5]:
# Add a new column
df_with_bonus = df.withColumn("bonus", col("salary") * 0.1)

In [6]:
# Replace an existing column
df_upper = df.withColumn("name", upper(col("name")))

In [8]:
df_with_bonus.show()
df_upper.show()

+------+-----+-----------+------+----------+--------+-------+
|emp_id| name| department|salary| hire_date|  status|  bonus|
+------+-----+-----------+------+----------+--------+-------+
|     1|alice|engineering| 95000|2022-03-15|  active| 9500.0|
|     2|  bob|  marketing| 72000|2021-07-01|  active| 7200.0|
|     3|carol|engineering|110000|2019-11-20|  active|11000.0|
|     4| dave|      sales| 68000|2023-01-10|inactive| 6800.0|
|     5|  eve|engineering|125000|2018-06-05|  active|12500.0|
|     6|frank|  marketing| 78000|2022-09-12|inactive| 7800.0|
+------+-----+-----------+------+----------+--------+-------+

+------+-----+-----------+------+----------+--------+
|emp_id| name| department|salary| hire_date|  status|
+------+-----+-----------+------+----------+--------+
|     1|ALICE|engineering| 95000|2022-03-15|  active|
|     2|  BOB|  marketing| 72000|2021-07-01|  active|
|     3|CAROL|engineering|110000|2019-11-20|  active|
|     4| DAVE|      sales| 68000|2023-01-10|inactive|
|

In [9]:
result = (
    df
    .withColumn("bonus", col("salary") * 0.1)
    .withColumn("total_comp", col("salary") + col("bonus"))
    .withColumn("name", upper(col("name")))
    .withColumn("hire_year", year(col("hire_date")))
)
result.show()

+------+-----+-----------+------+----------+--------+-------+----------+---------+
|emp_id| name| department|salary| hire_date|  status|  bonus|total_comp|hire_year|
+------+-----+-----------+------+----------+--------+-------+----------+---------+
|     1|ALICE|engineering| 95000|2022-03-15|  active| 9500.0|  104500.0|     2022|
|     2|  BOB|  marketing| 72000|2021-07-01|  active| 7200.0|   79200.0|     2021|
|     3|CAROL|engineering|110000|2019-11-20|  active|11000.0|  121000.0|     2019|
|     4| DAVE|      sales| 68000|2023-01-10|inactive| 6800.0|   74800.0|     2023|
|     5|  EVE|engineering|125000|2018-06-05|  active|12500.0|  137500.0|     2018|
|     6|FRANK|  marketing| 78000|2022-09-12|inactive| 7800.0|   85800.0|     2022|
+------+-----+-----------+------+----------+--------+-------+----------+---------+



In [10]:
df_with_band = df.withColumn(
    "salary_band",
    when(col("salary") >= 100000, "senior")
    .when(col("salary") >= 80000, "mid")
    .otherwise("junior")
)
df_with_band.select("name", "salary", "salary_band").show()

+-----+------+-----------+
| name|salary|salary_band|
+-----+------+-----------+
|alice| 95000|        mid|
|  bob| 72000|     junior|
|carol|110000|     senior|
| dave| 68000|     junior|
|  eve|125000|     senior|
|frank| 78000|     junior|
+-----+------+-----------+

